# 11. Bias in inference and mitigation: loan approval

The loan table from [WorkshopIgualdad2025](https://github.com/rferper/WorkshopIgualdad2025) is the one the workshop used to **mitigate** gender bias: drop the sensitive attribute, reweigh samples (Kamiran and Calders 2012), or penalise demographic parity in the genetic loss.

Those three ideas are now `ebdai.reweigh_weights`, `ebdai.weighted_mcc_loss` and `ebdai.fairness_regularized_loss`, and they plug into `ex_fuzzy.BaseFuzzyRulesClassifier.customized_loss`.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

from ex_fuzzy import BaseFuzzyRulesClassifier, FUZZY_SETS, eval_tools
from ebdai import (
    features_and_target, load_loan_approval, outcome_rates_by_group,
    plot_outcome_rates, fairness_report, reweigh_weights,
    weighted_mcc_loss, fairness_regularized_loss,
    parse_printed_rules, winning_rules_by_group,
    plot_winning_rules_by_group,
)

frame, sensitive = load_loan_approval()
X, y = features_and_target(frame, 'Loan_Status')
print(X.head())
rates = outcome_rates_by_group(y, X[sensitive], positive_label=1)
print(rates)
plot_outcome_rates(rates, title='Loan approval rate by gender')

## Baseline fuzzy rules (sensitive attribute kept)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)

def fit_rules(X_tr, y_tr, X_te, y_te, loss=None):
    clf = BaseFuzzyRulesClassifier(
        nRules=8, nAnts=3, fuzzy_type=FUZZY_SETS.t1,
        n_linguistic_variables=3, ds_mode=2, verbose=False,
        n_gen=6, pop_size=12, patience=3, random_state=42,
    )
    if loss is not None:
        clf.customized_loss(loss)
    clf.fit(X_tr, y_tr)
    report = eval_tools.eval_fuzzy_model(
        clf, X_tr, y_tr, X_te, y_te,
        plot_rules=False, print_rules=False, plot_partitions=False,
        return_rules=True, bootstrap_results_print=False,
    )
    return clf, report

clf, report = fit_rules(X_train, y_train, X_test, y_test)
y_pred = clf.predict(X_test)
table, gaps = fairness_report(y_test, y_pred, X_test[sensitive])
print('baseline')
print(table)
print(pd.Series(gaps))

## Reweighing

Kamiran and Calders weights make P(Y, A) independent in the training sample. Pass them through `weighted_mcc_loss` into `customized_loss`.

In [ ]:
weights = reweigh_weights(y_train, X_train[sensitive])
clf_w, _ = fit_rules(
    X_train, y_train, X_test, y_test,
    loss=weighted_mcc_loss(weights),
)
table_w, gaps_w = fairness_report(
    y_test, clf_w.predict(X_test), X_test[sensitive]
)
print('reweighed')
print(table_w)
print(pd.Series(gaps_w))

## Fairness regularisation

The genetic objective becomes MCC minus λ times the demographic-parity difference on the training rows (λ = 0.2, as in the workshop).

In [ ]:
clf_f, report_f = fit_rules(
    X_train, y_train, X_test, y_test,
    loss=fairness_regularized_loss(X_train[sensitive], lam=0.2),
)
table_f, gaps_f = fairness_report(
    y_test, clf_f.predict(X_test), X_test[sensitive]
)
print('regularised')
print(table_f)
print(pd.Series(gaps_f))
counts = winning_rules_by_group(
    clf_f, X_test, X_test[sensitive],
    rule_texts=parse_printed_rules(report_f or ''),
)
plot_winning_rules_by_group(
    counts, title='Winning loan rules by gender (regularised fit)'
)